### 뉴스 데이터 1000 개
- 본문을 읽고 정형 데이터로 변환하는 자동화 파이프 라인 만들기
- 1000개 데이터의 내용을 카테고리, 핵심어, 요약하면 
- 항목
    - 카테고리
    - 요약
    - 핵심어
-구조화된 출력

### 파이프라인 설계
1. 읽기 : 뉴스 본문 불러오기
2. 분석 : 각 기사를 정해진 항목으로 분석
3. 정리,저장  : 표로 모아서 csv저장

In [1]:
import pandas as pd
df = pd.read_csv("../data/11-1_뉴스정제.csv")
df = df.head(5)

In [9]:
## 셀 1. 라이브러리와 API 준비

# JSON 문자열을 Python 딕셔너리로 변환하기 위해 사용
import json

# 진행 상황을 잠시 멈추기 위해 사용
import time

# 데이터프레임을 만들고 CSV를 읽고 저장하기 위해 사용
import pandas as pd

# 환경변수를 불러오기 위해 사용
from dotenv import load_dotenv

# OpenAI API 사용
from openai import OpenAI

# .env 파일에 있는 OPENAI_API_KEY 불러오기
load_dotenv()

# OpenAI 클라이언트 생성
client = OpenAI()

# 사용할 모델명
MODEL = "gpt-5.6-luna"

In [10]:
## 셀 2. 뉴스 데이터 읽기

# 뉴스 CSV 파일 읽기
df = pd.read_csv("../data/11-1_뉴스정제.csv")

# 전체 행과 열 개수 확인
print("데이터 크기:", df.shape)

# 컬럼명 확인
print("컬럼명:", list(df.columns))

# 데이터 앞부분 확인
display(df.head())

# 현재 데이터는 이미 1000개이므로 다음과 같이 확인할 수 있습니다.

# 뉴스 데이터 개수 확인
print(f"전체 뉴스 개수: {len(df)}개")

데이터 크기: (1000, 6)
컬럼명: ['제목', '본문', '카테고리', '요약', '출처URL', '정제본문']


,제목,본문,카테고리,요약,출처URL,정제본문
0,현대백화점그룹 더현대 광주 추진,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...,경제,"6 6일 현대백화점그룹이 광주시에 문화복합몰을 만든다고 6일 밝혔으며, 광주시는 서...",https://n.news.naver.com/mnews/article/001/001...,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...
1,이스타항공 이상직 회사와 무관…오해 살 언동 말아야,전주 뉴시스 김얼 기자 이스타항공 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직...,경제,이이스항공은 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직 전 의원이 출소한 것...,https://n.news.naver.com/mnews/article/003/001...,전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직...
2,농협은행 농협금융 출범 10주년 기념주화 NFT 이벤트,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 ‘10주년 기념주...,경제,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 소셜미디어 인스타...,https://n.news.naver.com/mnews/article/366/000...,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 10주년 기념주화...
3,오늘부터 유류세 인하 폭 확대…하반기 바뀌는 세제·금융 정책은,img tag s 지난 30일 서울의 한 주유소. 〈사진 연합뉴스〉 img tag ...,경제,정부는 고유가 상황에 따라 국민의 유류비 부담 완화를 위해 이날부터 유류세를 법정 ...,https://n.news.naver.com/mnews/article/437/000...,img tag s 지난 30일 서울의 한 주유소 사진 연합뉴스 img tag e 오...
4,푸르덴셜생명 더 큰 드림 변액연금보험Ⅱ에 신규펀드 13종 추가,파이낸셜뉴스 푸르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림...,경제,지난르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림 변액연금보...,https://n.news.naver.com/mnews/article/014/000...,파이낸셜뉴스 푸르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림...


전체 뉴스 개수: 1000개


In [11]:
## 셀 3. 분석할 본문 컬럼 선택

# 정제본문 컬럼이 있으면 정제본문을 사용
# 없으면 본문 컬럼을 사용
if "정제본문" in df.columns:
    body_column = "정제본문"
else:
    body_column = "본문"

print("분석에 사용할 본문 컬럼:", body_column)

분석에 사용할 본문 컬럼: 정제본문


In [ ]:
## 셀 4. 뉴스 분석 프롬프트 만들기
def make_news_prompt(title, body):
    # 뉴스 제목과 본문을 프롬프트에 삽입
    prompt = f"""
다음 뉴스 기사를 분석하고 JSON 형식으로만 응답해줘.

[뉴스 제목]
{title}

[뉴스 본문]
{body}

다음 구조를 반드시 지켜줘.

{{
    "category": "카테고리",
    "summary": "뉴스 요약",
    "keywords": ["위기", "폭락", "악재"]
}}

작성 규칙:
- category는 경제, 정치, 사회, IT·과학, 문화, 스포츠, 국제, 기타 중 하나로 작성
- summary는 뉴스의 핵심 내용을 2문장 이내로 작성
- keywords는 핵심어 3~5개를 리스트로 작성
- 뉴스 본문에 없는 내용은 추측하지 않기
- JSON 이외의 설명은 작성하지 않기
"""

    # 완성된 프롬프트 반환
    return prompt

In [ ]:
## 셀 5. 뉴스 1개 분석 함수 만들기

def analyze_news(title, body):
# 뉴스 제목과 본문을 포함한 프롬프트 생성
prompt = make_news_prompt(title, body)

# OpenAI API에 분석 요청
response = client.chat.completions.create(
    model=MODEL,
    reasoning_effort="none",

    # JSON 형태로 응답하도록 요청
    response_format={
        "type": "json_object"
    },

    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

# 모델의 답변 내용 가져오기
content = response.choices[0].message.content

# JSON 문자열을 Python 딕셔너리로 변환
result = json.loads(content)

# 결과에 필요한 항목이 모두 있는지 확인
required_keys = [
    "category",
    "summary",
    "keywords"
]

for key in required_keys:
    if key not in result:
        raise ValueError(f"{key} 항목이 없습니다.")

# 분석 결과 반환
return result

In [ ]:

## 셀 6. 뉴스 1개 테스트

# 첫 번째 뉴스 1개 선택
test_row = df.iloc[0]

# 뉴스 1개 분석
test_result = analyze_news(
    title=str(test_row["제목"]),
    body=str(test_row[body_column])
)

# 분석 결과 출력
print(json.dumps(
    test_result,
    ensure_ascii=False,
    indent=2
))

# 예상 결과는 다음과 같은 구조입니다.

{
"category": "경제",
"summary": "뉴스의 핵심 내용입니다.",
"keywords": ["기업", "투자", "시장"]
}

In [ ]:
## 셀 7. 5개 뉴스로 반복 테스트

# 처음에는 5개만 테스트
test_df = df.head(5)

# 분석 결과를 저장할 빈 리스트
results = []

# 뉴스 5개를 하나씩 반복
for index, row in test_df.iterrows():
    try:
        # 뉴스 제목 가져오기
        title = str(row["제목"])

        # 뉴스 본문 가져오기
        body = str(row[body_column])

        # 뉴스 분석 실행
        analysis = analyze_news(title, body)

        # 기존 데이터와 분석 결과를 하나로 묶기
        results.append({
            "원본인덱스": index,
            "제목": title,
            "본문": body,
            "카테고리": analysis["category"],
            "요약": analysis["summary"],
            "핵심어": ", ".join(analysis["keywords"]),
            "출처URL": row.get("출처URL", ""),
            "처리상태": "성공",
            "오류": ""
        })

        # 진행 상황 출력
        print(f"{index + 1}/{len(test_df)} 처리 완료")

        # API 요청 간격
        time.sleep(0.2)

    except Exception as error:
        # 실패한 데이터도 결과에 기록
        results.append({
            "원본인덱스": index,
            "제목": row.get("제목", ""),
            "본문": row.get(body_column, ""),
            "카테고리": "",
            "요약": "",
            "핵심어": "",
            "출처URL": row.get("출처URL", ""),
            "처리상태": "실패",
            "오류": str(error)
        })

        print(f"{index + 1}/{len(test_df)} 처리 실패")
        print(error)

In [ ]:
## 셀 8. 결과를 표로 만들기

# 리스트 형태의 분석 결과를 데이터프레임으로 변환
result_df = pd.DataFrame(results)

# 결과 확인
display(result_df)

## 셀 9. CSV 파일로 저장하기

# 분석 결과를 CSV 파일로 저장
result_df.to_csv(
    "news_analysis_test.csv",
    index=False,
    encoding="utf-8-sig"
)

print("테스트 결과 저장 완료")



In [ ]:
## 셀 10. 전체 1000개 처리
# 5개 테스트가 정상 작동한 뒤 아래 코드를 실행하세요.
# 전체 뉴스 데이터를 사용
target_df = df

# 분석 결과를 저장할 리스트
results = []

# 전체 뉴스 반복 처리
for index, row in target_df.iterrows():
    try:
        # 제목과 본문 가져오기
        title = str(row["제목"])
        body = str(row[body_column])

        # LLM으로 뉴스 분석
        analysis = analyze_news(title, body)

        # 분석 결과 저장
        results.append({
            "원본인덱스": index,
            "제목": title,
            "본문": body,
            "카테고리": analysis["category"],
            "요약": analysis["summary"],
            "핵심어": ", ".join(analysis["keywords"]),
            "출처URL": row.get("출처URL", ""),
            "처리상태": "성공",
            "오류": ""
        })

        # 진행 상황 출력
        print(f"{index + 1}/{len(target_df)} 처리 완료")

        # API 호출 간격
        time.sleep(0.2)

    except Exception as error:
        # 실패한 뉴스도 기록
        results.append({
            "원본인덱스": index,
            "제목": row.get("제목", ""),
            "본문": row.get(body_column, ""),
            "카테고리": "",
            "요약": "",
            "핵심어": "",
            "출처URL": row.get("출처URL", ""),
            "처리상태": "실패",
            "오류": str(error)
        })

        print(f"{index + 1}/{len(target_df)} 처리 실패:", error)



In [ ]:
## 셀 11. 최종 결과 저장

# 분석 결과를 데이터프레임으로 변환
result_df = pd.DataFrame(results)

# 최종 CSV 파일 저장
result_df.to_csv(
"news_analysis_result.csv",
index=False,
encoding="utf-8-sig"
)

# 처리 결과 확인
print("최종 처리 완료")
print(f"전체 결과: {len(result_df)}개")
print(f"성공: {(result_df['처리상태'] == '성공').sum()}개")
print(f"실패: {(result_df['처리상태'] == '실패').sum()}개")

display(result_df.head())

# 이 노트북에서 기억할 핵심은 다음입니다.

make_news_prompt()
→ 분석 요청문 만들기

analyze_news()
→ 뉴스 1개 분석하기

for 반복문
→ 뉴스 1000개 분석하기

pd.DataFrame()
→ 결과를 표로 만들기

to_csv()
→ CSV 파일로 저장하기